# **Generate Reversal ABCDEF Multi-Timestep Task Sequences**

Generates two 6-stimulus reversal task sequences with multi-timestep trial structure.

**Stimulus groups:**
- Group 1 (A, B): 100% reward probability pre-reversal
- Group 2 (C, D): 50% reward probability — never reverses
- Group 3 (E, F): 0% reward probability pre-reversal

**Reversal variants:**
1. **Full reversal** — both A and B swap reward with both E and F
   - Pre:  A=100%, B=100%, C=50%, D=50%, E=0%, F=0%
   - Post: A=0%, B=0%, C=50%, D=50%, E=100%, F=100%
2. **Partial reversal** — only A and E swap; B and F keep their contingencies
   - Pre:  A=100%, B=100%, C=50%, D=50%, E=0%, F=0%
   - Post: A=0%, B=100%, C=50%, D=50%, E=100%, F=0%

**Trial Structure:**
- Stimulus window: multiple timesteps showing the stimulus
- Reward window: multiple timesteps where reward can be obtained
- ITI: random inter-trial interval

Output files:
- `task_data/reversal_abcdef_multitimestep_full.pkl`
- `task_data/reversal_abcdef_multitimestep_partial.pkl`

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Task Parameters

In [ ]:
# Stimulus identities
stimuli = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5}
rewards = {"no_reward": 0, "reward": 1}

# Task parameters
num_pre_reversal_trials  = 4000
num_post_reversal_trials = 4000

stim_window = 5    # timesteps showing stimulus
reward_window = 3  # timesteps in reward-availability window
min_iti = 10
max_iti = 20

seed = 42
np.random.seed(seed)

# State map (indices for one-hot encoding)
state_map = {
    "A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5,
    "reward_unknown": 6, "unrewarded": 7, "rewarded": 8, "ITI": 9
}
STATE_DIM = 10

print(f"Pre-reversal trials:  {num_pre_reversal_trials}")
print(f"Post-reversal trials: {num_post_reversal_trials}")
print(f"Stimulus window:      {stim_window} timesteps")
print(f"Reward window:        {reward_window} timesteps")
print(f"ITI range:            {min_iti}-{max_iti} timesteps")

---
## Helper: reward probability look-up tables

These dicts map stimulus index → reward probability for each phase.

In [ ]:
# Pre-reversal contingencies (same for both reversal types)
reward_prob_pre = {
    stimuli["A"]: 1.0,   # 100%
    stimuli["B"]: 1.0,   # 100%
    stimuli["C"]: 0.5,   # 50%
    stimuli["D"]: 0.5,   # 50%
    stimuli["E"]: 0.0,   # 0%
    stimuli["F"]: 0.0,   # 0%
}

# Post-reversal contingencies — FULL reversal
reward_prob_post_full = {
    stimuli["A"]: 0.0,   # was 100% → now 0%
    stimuli["B"]: 0.0,   # was 100% → now 0%
    stimuli["C"]: 0.5,   # unchanged
    stimuli["D"]: 0.5,   # unchanged
    stimuli["E"]: 1.0,   # was 0% → now 100%
    stimuli["F"]: 1.0,   # was 0% → now 100%
}

# Post-reversal contingencies — PARTIAL reversal (A↔E swap, B and F unchanged)
reward_prob_post_partial = {
    stimuli["A"]: 0.0,   # was 100% → now 0%
    stimuli["B"]: 1.0,   # unchanged 100%
    stimuli["C"]: 0.5,   # unchanged
    stimuli["D"]: 0.5,   # unchanged
    stimuli["E"]: 1.0,   # was 0% → now 100%
    stimuli["F"]: 0.0,   # unchanged 0%
}

print("Pre-reversal reward probabilities:")
for name, idx in stimuli.items():
    print(f"  {name}: {reward_prob_pre[idx]*100:.0f}%")

---
## Helper: generate trial sequence

In [ ]:
def sample_reward(prob):
    """Return 1 (reward) with probability `prob`, else 0."""
    return 1 if np.random.rand() < prob else 0


def generate_trial_data(num_pre, num_post, reward_prob_post, seed=42):
    """Generate trial-level stimulus and reward data."""
    np.random.seed(seed)
    all_stimuli = list(stimuli.values())  # [0,1,2,3,4,5]

    trial_data = {"stimuli": [], "rewards": [], "masks": {"reversal": []}}

    for phase, num_trials, reward_prob in [
        (0, num_pre,  reward_prob_pre),
        (1, num_post, reward_prob_post),
    ]:
        for _ in range(num_trials):
            stim = np.random.choice(all_stimuli)
            rew  = sample_reward(reward_prob[stim])
            trial_data["stimuli"].append(stim)
            trial_data["rewards"].append(rew)
            trial_data["masks"]["reversal"].append(phase)

    return trial_data


def expand_to_timesteps(trial_data):
    """Expand trial-level data to timestep-level sequences."""
    state_sequence = []
    reward_sequence = []
    trial_structure = []

    trial_idx = 0
    timestep  = 0

    for stim, reward_avail, reversal_phase in zip(
        trial_data["stimuli"],
        trial_data["rewards"],
        trial_data["masks"]["reversal"]
    ):
        trial_start_timestep = timestep

        # --- stimulus window ---
        stim_timesteps = []
        for _ in range(stim_window):
            state_sequence.append(stim)
            reward_sequence.append(0.0)
            stim_timesteps.append(timestep)
            timestep += 1

        # --- reward window (reward_unknown state) ---
        reward_timesteps = []
        for _ in range(reward_window):
            state_sequence.append(state_map["reward_unknown"])
            reward_sequence.append(float(reward_avail == rewards["reward"]))
            reward_timesteps.append(timestep)
            timestep += 1

        # --- ITI ---
        iti_length = np.random.randint(min_iti, max_iti + 1)
        iti_timesteps = []
        for _ in range(iti_length):
            state_sequence.append(state_map["ITI"])
            reward_sequence.append(0.0)
            iti_timesteps.append(timestep)
            timestep += 1

        trial_structure.append({
            "trial_idx": trial_idx,
            "stimulus": stim,
            "reward_available": (reward_avail == rewards["reward"]),
            "reversal_phase": reversal_phase,
            "trial_start": trial_start_timestep,
            "stim_window":   stim_timesteps,
            "reward_window": reward_timesteps,
            "iti_window":    iti_timesteps,
            "trial_end": timestep - 1,
        })
        trial_idx += 1

    return state_sequence, reward_sequence, trial_structure, timestep


def to_ohe(state_sequence, state_dim=STATE_DIM):
    """Convert integer state list to one-hot array of shape (N, state_dim)."""
    ohe = np.zeros((len(state_sequence), state_dim), dtype=np.float32)
    for i, s in enumerate(state_sequence):
        if 0 <= s < state_dim:
            ohe[i, s] = 1.0
    return ohe


def make_phase_boundaries(trial_structure, num_pre):
    """Build phase_boundaries dict from trial structure."""
    reversal_point = trial_structure[num_pre - 1]["trial_end"] + 1
    total = trial_structure[-1]["trial_end"] + 1
    return {
        "reversal_points": [reversal_point],
        "pre_reversal":  {"start": 0,              "end": reversal_point},
        "post_reversal": {"start": reversal_point, "end": total},
    }


def assemble_data(trial_data, state_sequence, reward_sequence,
                  trial_structure, num_pre):
    """Assemble everything into the standard data dict."""
    state_sequence_ohe = to_ohe(state_sequence)
    phase_boundaries   = make_phase_boundaries(trial_structure, num_pre)
    return {
        "state_sequence_ohe": state_sequence_ohe,
        "reward_sequence":    np.array(reward_sequence, dtype=np.float32),
        "sequence": {
            "stimuli": trial_data["stimuli"],
            "rewards": trial_data["rewards"],
            "masks":   trial_data["masks"],
        },
        "phase_boundaries": phase_boundaries,
        "trial_structure":  trial_structure,
        "state_map":        state_map,
        "trial_params": {
            "stim_window":  stim_window,
            "reward_window": reward_window,
            "min_iti": min_iti,
            "max_iti": max_iti,
        },
        "reversal_type": None,  # filled in below
    }

print("Helper functions defined.")

---
## Generate: Full Reversal

In [ ]:
np.random.seed(seed)
td_full = generate_trial_data(
    num_pre_reversal_trials, num_post_reversal_trials,
    reward_prob_post_full, seed=seed
)
ss_full, rs_full, ts_full, _ = expand_to_timesteps(td_full)
data_full = assemble_data(td_full, ss_full, rs_full, ts_full, num_pre_reversal_trials)
data_full["reversal_type"] = "full"

print(f"Full reversal — total timesteps: {len(data_full['state_sequence_ohe'])}")
print(f"  Total trials: {len(ts_full)}")
print(f"  Pre-reversal trials:  {num_pre_reversal_trials}")
print(f"  Post-reversal trials: {num_post_reversal_trials}")
print(f"  State sequence shape: {data_full['state_sequence_ohe'].shape}")
print(f"  Reversal point (timestep): {data_full['phase_boundaries']['reversal_points'][0]}")

---
## Generate: Partial Reversal

In [ ]:
np.random.seed(seed)
td_partial = generate_trial_data(
    num_pre_reversal_trials, num_post_reversal_trials,
    reward_prob_post_partial, seed=seed
)
ss_partial, rs_partial, ts_partial, _ = expand_to_timesteps(td_partial)
data_partial = assemble_data(td_partial, ss_partial, rs_partial, ts_partial, num_pre_reversal_trials)
data_partial["reversal_type"] = "partial"

print(f"Partial reversal — total timesteps: {len(data_partial['state_sequence_ohe'])}")
print(f"  State sequence shape: {data_partial['state_sequence_ohe'].shape}")
print(f"  Reversal point (timestep): {data_partial['phase_boundaries']['reversal_points'][0]}")

---
## Verify reward contingencies

In [ ]:
stim_names = ["A", "B", "C", "D", "E", "F"]

for label, data in [("Full", data_full), ("Partial", data_partial)]:
    print(f"\n=== {label} reversal ===")
    for phase_label, phase_idx in [("Pre", 0), ("Post", 1)]:
        phase_trials = [t for t in data["trial_structure"] if t["reversal_phase"] == phase_idx]
        print(f"  {phase_label}-reversal:")
        for s_idx, s_name in enumerate(stim_names):
            s_trials = [t for t in phase_trials if t["stimulus"] == s_idx]
            if s_trials:
                rew_rate = np.mean([t["reward_available"] for t in s_trials])
                print(f"    {s_name}: {rew_rate*100:.1f}% rewarded  (n={len(s_trials)})")

---
## Save to pickle files

In [ ]:
output_dir = Path("/Users/pmccarthy/Documents/cogNN/task_data")
output_dir.mkdir(parents=True, exist_ok=True)

path_full    = output_dir / "reversal_abcdef_multitimestep_full.pkl"
path_partial = output_dir / "reversal_abcdef_multitimestep_partial.pkl"

with open(path_full, "wb") as f:
    pickle.dump(data_full, f)
print(f"Saved full reversal to:    {path_full}")

with open(path_partial, "wb") as f:
    pickle.dump(data_partial, f)
print(f"Saved partial reversal to: {path_partial}")

---
## Quick visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3", "#a65628"]

for ax, (label, data) in zip(axes, [("Full reversal", data_full), ("Partial reversal", data_partial)]):
    window = 200  # running average window in trials
    for s_idx, s_name in enumerate(stim_names):
        s_trials  = [i for i, t in enumerate(data["trial_structure"]) if t["stimulus"] == s_idx]
        s_rewards = [data["trial_structure"][i]["reward_available"] for i in s_trials]
        # Running mean
        rm = np.convolve(s_rewards, np.ones(window) / window, mode="valid")
        trial_x = np.array(s_trials[window - 1:])
        ax.plot(trial_x, rm, color=colors[s_idx], label=s_name, lw=1.5)

    # Reversal line
    ax.axvline(num_pre_reversal_trials, color="k", ls="--", lw=1, label="Reversal")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Reward probability")
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()